In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction(tx_number):
    return {
        "tx_id":    f"TX{tx_number:04d}",
        "user_id":  f"u{random.randint(1, 20):02d}",
        "amount":   round(random.uniform(5.0, 5000.0), 2),
        "store":    random.choice(["Warszawa", "Kraków", "Gdańsk", "Wrocław"]),
        "category": random.choice(["elektronika", "odzież", "żywność", "książki"]),
        "timestamp": datetime.now().isoformat()
    }

tx_counter = 1
while True:
    transaction = generate_transaction(tx_counter)
    producer.send('transactions', value=transaction)
    print(f"[{transaction['timestamp']}] Wysłano: {transaction}")
    tx_counter += 1
    time.sleep(1)

Writing producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    if message.value > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Writing consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrichment-group',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

def assign_risk_level(amount):
    if amount > 3000:
        return "HIGH"
    elif amount > 1000:
        return "MEDIUM"
    else:
        return "LOW"

for message in consumer:
    tx = message.value
    tx['risk_level'] = assign_risk_level(tx['amount'])
    print(f"[{tx['risk_level']:6}] {tx['tx_id']} | {tx['amount']:8.2f} PLN | {tx['store']:<10} | {tx['category']}")

Writing consumer_enrich.py


In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='analytics-group',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

def print_summary():
    print(f"\n{'─'*52}")
    print(f"{'Sklep':<12} {'Liczba':>7} {'Suma':>12} {'Średnia':>12}")
    print(f"{'─'*52}")
    for store in sorted(store_counts):
        count  = store_counts[store]
        total  = total_amount[store]
        avg    = total / count
        print(f"{store:<12} {count:>7} {total:>11.2f} {avg:>12.2f}")
    print(f"{'─'*52}")

for message in consumer:
    tx = message.value
    store = tx['store']

    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0.0) + tx['amount']
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"[po {msg_count} wiadomościach]")
        print_summary()

Writing consumer_count.py


In [5]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='stats-group',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    'count':   0,
    'total':   0.0,
    'min':     float('inf'),
    'max':     float('-inf')
})

msg_count = 0

def print_stats():
    print(f"\n{'─'*62}")
    print(f"{'Kategoria':<14} {'Liczba':>7} {'Przychód':>12} {'Min':>10} {'Max':>10}")
    print(f"{'─'*62}")
    for cat in sorted(stats):
        s = stats[cat]
        print(f"{cat:<14} {s['count']:>7} {s['total']:>11.2f} {s['min']:>10.2f} {s['max']:>10.2f}")
    print(f"{'─'*62}")

for message in consumer:
    tx    = message.value
    cat   = tx['category']
    amount = tx['amount']

    stats[cat]['count'] += 1
    stats[cat]['total'] += amount
    stats[cat]['min']    = min(stats[cat]['min'], amount)
    stats[cat]['max']    = max(stats[cat]['max'], amount)
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"[po {msg_count} wiadomościach]")
        print_stats()

Writing consumer_stats.py


In [6]:
%%file consumer_velocity.py
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='velocity-group',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Dla każdego user_id przechowujemy listę timestampów ostatnich transakcji
user_timestamps = defaultdict(list)
WINDOW_SECONDS = 60
THRESHOLD = 3

def get_transactions_in_window(timestamps, now):
    cutoff = now.timestamp() - WINDOW_SECONDS
    return [t for t in timestamps if t >= cutoff]

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    now = datetime.fromisoformat(tx['timestamp'])
    
    recent = get_transactions_in_window(user_timestamps[user_id], now)
    recent.append(now.timestamp())
    user_timestamps[user_id] = recent

    count = len(recent)

    if count > THRESHOLD:
        print(
            f"ALERT| USER: {user_id} | "
            f"{count} transakcji w 60s"
        )

Writing consumer_velocity.py
